In [0]:
source_data_path = "/Volumes/workspace/default/source_data/source_data.csv"

# Bronze
Requirements:
- all records and columns ingested

all records and columns ingested

In [0]:
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("nullValue", "")\
    .load(source_data_path)

df.write.format("delta").mode("overwrite").saveAsTable("bronze_data")
display(df)

# Silver
Requirements:
- contains only full records
- contains only valid records
- column names expanded where necessary

In [0]:
df = spark.table("bronze_data")

In [0]:
import pandas as pd

# drop columns that exceed the missing value percentage defined in the threshold

THRESHOLD = 90 

column_stats = []
columns_to_drop = []

df_count = df.count()

for column in df.columns:

    col_count = df.filter(df[column].isNull()).count()
    percentage = (col_count/df_count)*100
    drop = percentage >= THRESHOLD

    column_stats.append({
        "name": column,
        "number_of_missing_values": col_count,
        "percentage_of_missing_values": percentage,
        "drop": drop})
    
    if drop:
        columns_to_drop.append(column)

column_stats_df = pd.DataFrame(column_stats)

print("Summary of missing values by column:")
display(column_stats_df.sort_values(by="number_of_missing_values", ascending=False))

df = df.drop(*columns_to_drop)

print(f"Dropped columns:\n{',\n'.join(columns_to_drop)}\n")
print(f"Remaining columns:\n{',\n'.join(df.columns)}")


In [0]:
# drop rows containing missing values

temp_df_count = df.count()

df = df.dropna()

print(f"Number of dropped rows: {temp_df_count - df.count()}")
print("Remaining rows:")
display(df)

In [0]:
# expand column name where necessary

rename_map = {
    "oph": "operating_hours",
    "pist_m": "piston_material",
    "issue_type": "combustion_issue_type",
    "bmep": "break_mean_effective_pressure",
    "ng_imp": "natural_gas_impurities_nmol",
    "past_dmg": "had_past_damages",
    "rpm_max": "maximum_rotations_per_minute",
    "full_load_issues": "had_full_load_issues",
    "number_up": "number_of_unplanned_events",
    "number_tc": "number_of_turbo_chargers",
    "op_set_1": "operational_setting_1",
    "op_set_2": "operational_setting_2",
    "op_set_3": "operational_setting_3",
}

for old_name, new_name in rename_map.items():
    df = df.withColumnRenamed(old_name, new_name)

display(df)

In [0]:
df.write.format("delta").mode("overwrite").saveAsTable("silver_data")

#Gold 
Requirements:
- keeps columns that contain information
- keeps records that passed **input** and **business tests**

**Input tests**:
- records with mismatching types 
- records with invalid values
- records with missing values

**Business tests**:
- OPH <= 120000

In [0]:
df = spark.table("silver_data")

In [0]:
# drop columns that contains only just a single value 

columns_to_drop = []

for column in df.columns:
        
    if df.select(column).distinct().count() == 1:
        columns_to_drop.append(column)

df = df.drop(*columns_to_drop)

print(f"Dropped columns:\n{',\n'.join(columns_to_drop)}\n")
print(f"Remaining columns:\n{',\n'.join(df.columns)}")


In [0]:
# cast columns to their expected data types and drop rows where casting was unsuccessful

type_map = {
    "operating_hours": "integer",            
    "piston_material": "integer",            
    "issue_type": "string",     
    "bmep": "double",          
    "natural_gas_impurities_nmol": "double",         
    "past_damages": "boolean",          
    "resting_analysis_results": "string",
    "rpm_max": "double",
    "full_load_issues": "boolean",
    "number_of_unplanned_events": "integer",
    "number_of_turbo_chargers": "integer",
    "operational_setting_1": "double",
    "operational_setting_2": "double",
    "operational_setting_3": "double",
    "high_breakdown_risk": "boolean"
}

for column, cast_type in type_map.items():
    if column in df.columns:
        df = df.withColumn(column, df[column].try_cast(cast_type))

print("New schema:")
df.printSchema()

temp_df_count = df.count()

df = df.dropna()

print(f"Number of dropped rows: {temp_df_count - df.count()}")
print("Remaining rows:")
display(df)

In [0]:
# replace category names and drop records with invalid values

category_map = {           
    "issue_type": {
        "1": "typical",
        "2": "atypical", 
        "3": "non-related", 
        "4": "non-symptomatic"},              
    "resting_analysis_results": {
        "0": "normal", 
        "1": "abnormal", 
        "2": "critical"},
}

temp_df_count = df.count()

for column, mapping in category_map.items():
    if column in df.columns:
        df = df.replace(to_replace=mapping, subset=[column])
        
        valid_values = list(mapping.values())

        df = df.filter(df[column].isin(valid_values))
        
print(f"Number of dropped rows: {temp_df_count - df.count()}")
print("Remaining rows:")
display(df)

In [0]:
temp_df_count = df.count()

df = df.filter(df["operating_hours"] <= 120000)

print(f"Number of dropped rows: {temp_df_count - df.count()}")
print("Remaining rows:")
display(df)

In [0]:
df.write.format("delta").mode("overwrite").saveAsTable("gold_data")